In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('../data/raw/ai4i2020.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [23]:
# =====================================================================
# AI4I 2020 Predictive Maintenance Dataset - 欄位定義對照表 (df.columns)
# =====================================================================

# 1. 識別資料 (Identification)
# df['UDI']              # 唯一識別碼 (Unique Device Identifier)：資料流水號 (1 ~ 10,000)
# df['Product ID']       # 產品 ID (Product ID)：產品的型號與序號編碼 (例如: L47181)
# df['Type']             # 產品類型 (Type)：品質等級，分為 L (低階 50%)、M (中階 30%)、H (高階 20%)

# 2. 感測器特徵數據 (Features - AI 模型輸入)
# df['Air temperature [K]']      # 環境空氣溫度 (Kelvin)：廠房內的室溫 (300K 約等於 27°C)
# df['Process temperature [K]']  # 加工運作溫度 (Kelvin)：機台運作時的內部實際溫度
# df['Rotational speed [rpm]']   # 轉速 (Revolutions Per Minute)：馬達每分鐘旋轉圈數
# df['Torque [Nm]']              # 扭力 (Newton-meters)：馬達旋轉時的出力大小/力矩
# df['Tool wear [min]']          # 刀具磨損時間 (Minutes)：目前刀具已經連續使用的累積時間

# 3. 預測目標與故障標籤 (Labels - AI 模型輸出)
# df['Machine failure']  # 機器故障總標籤：0 代表正常，1 代表發生故障 (主要預測目標)

# 4. 具體故障原因細分 (5大模式，當 Machine failure = 1 時觸發)
# df['TWF']              # 刀具磨損故障 (Tool Wear Failure)：刀具達到磨損極限或斷裂
# df['HDF']              # 熱散失故障 (Heat Dissipation Failure)：工作溫差與轉速異常導致散熱不良
# df['PWF']              # 電力功率故障 (Power Failure)：轉速與扭力乘積(功率)過高或過低導致卡死
# df['OSF']              # 過載負荷故障 (Overstrain Failure)：磨損時間與扭力乘積超過機件負荷極限
# df['RNF']              # 隨機故障 (Random Failure)：與參數無關的未知隨機錯誤
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [24]:
print(f'The description of dataset: \n{df.describe()}')
print(f'The shape of df: {df.shape}')

The description of dataset: 
               UDI  Air temperature [K]  Process temperature [K]  \
count  10000.00000         10000.000000             10000.000000   
mean    5000.50000           300.004930               310.005560   
std     2886.89568             2.000259                 1.483734   
min        1.00000           295.300000               305.700000   
25%     2500.75000           298.300000               308.800000   
50%     5000.50000           300.100000               310.100000   
75%     7500.25000           301.500000               311.100000   
max    10000.00000           304.500000               313.800000   

       Rotational speed [rpm]   Torque [Nm]  Tool wear [min]  Machine failure  \
count            10000.000000  10000.000000     10000.000000     10000.000000   
mean              1538.776100     39.986910       107.951000         0.033900   
std                179.284096      9.968934        63.654147         0.180981   
min               1168.000000     

In [30]:
product_ids = df.iloc[:30, :]
print(f'Selected product id: \n{product_ids.head()}')

Selected product id: 
   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    